In [ ]:
#| default_exp features.breakpoint_motif

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import numpy as np
import structlog
import re
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()
        

In [ ]:
#| export
def _parse_array(s):
    if not isinstance(s, str) or not s.startswith('['): 
        return []
    clean = s.replace('[', '').replace(']', '').replace(chr(10), '').replace(chr(13), '')
    try:
        return [float(x) for x in clean.split()]
    except:
        return []

class BreakPointMotifEvaluator(FeatureEvaluator):
    """Extracts 4-mer adjacent breakpoint motifs."""
    
    name = "BreakPointMotif"
    source_file = ".BreakPointMotif.ontarget.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            self.tier = 3
            self.category = "motifs"
            if "Motif" in cols and "Frequency" in cols:
                for _, row in df.iterrows():
                    m = str(row["Motif"])
                    if pd.notna(row["Frequency"]):
                        extracted[f"bpmotif_{m}"] = float(row["Frequency"])
    
            return extracted
        except Exception as e:
            log.warning("breakpoint_motif_extraction_failed", error=str(e))
            return {}


In [ ]:
#| test
from kreview.features.breakpoint_motif import _parse_array
evaluator = BreakPointMotifEvaluator()
df = pd.DataFrame([{"Motif": "ATCG", "Frequency": 0.1}])
res = evaluator.extract(df)
assert res["bpmotif_ATCG"] == 0.1
